In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

In [2]:
df = pd.read_csv("D:\Data Analysis Projects\CSV files\customer_churn_dataset.csv")

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Shape: 20000 rows × 11 columns


,customer_id,tenure,monthly_charges,total_charges,contract,payment_method,internet_service,tech_support,online_security,support_calls,churn
0,1,52,54.20,2818.40,Month-to-month,Credit,DSL,No,Yes,1,No
1,2,15,35.28,529.20,Month-to-month,Debit,DSL,No,No,2,No
2,3,72,78.24,5633.28,Month-to-month,Debit,DSL,No,No,0,No
3,4,61,80.24,4894.64,One year,Cash,Fiber,Yes,Yes,0,No
4,5,21,39.38,826.98,Month-to-month,UPI,Fiber,No,No,4,Yes


In [3]:
print("=== Data Types ===")
print(df.dtypes)

=== Data Types ===
customer_id           int64
tenure                int64
monthly_charges     float64
total_charges       float64
contract             object
payment_method       object
internet_service     object
tech_support         object
online_security      object
support_calls         int64
churn                object
dtype: object


In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_%': missing_pct
}).query('missing_count > 0')

print(missing_report if not missing_report.empty else "No missing values found ✓")

                  missing_count  missing_%
internet_service           2013      10.06


Investigate Missing Values

In [16]:
missing_mask = df['internet_service'].isna()

print(f"Missing internet_service: {missing_mask.sum()} rows ({missing_mask.mean()*100:.1f}%)\n")

# Do missing rows churn at a different rate?
print("=== Churn rate: missing vs present ===")
print(df.groupby(missing_mask)['churn']
        .apply(lambda x: (x == 'Yes').mean() * 100)
        .rename({False: 'has value', True: 'missing'})
        .round(2)
        .to_string())

# Any pattern with other columns?
print("\n=== Support calls (mean) by missing flag ===")
print(df.groupby(missing_mask)['support_calls'].mean().rename({False: 'has value', True: 'missing'}))

print("\n=== Contract type distribution in missing rows ===")
print(df[missing_mask]['contract'].value_counts())

Missing internet_service: 2013 rows (10.1%)

=== Churn rate: missing vs present ===
internet_service
has value   34.10
missing     35.22

=== Support calls (mean) by missing flag ===
internet_service
has value   1.51
missing     1.50
Name: support_calls, dtype: float64

=== Contract type distribution in missing rows ===
contract
Month-to-month    1222
One year           497
Two year           294
Name: count, dtype: int64


Fill Missing values

In [17]:
mode_val = df['internet_service'].mode()[0]
df['internet_service'] = df['internet_service'].fillna(mode_val)

print(f"Filled with mode: '{mode_val}'")
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print()
print("=== internet_service distribution after fill ===")
print(df['internet_service'].value_counts())
print(df['internet_service'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

Filled with mode: 'Fiber'
Missing values remaining: 0

=== internet_service distribution after fill ===
internet_service
Fiber    12077
DSL       7923
Name: count, dtype: int64
internet_service
Fiber    60.4%
DSL      39.6%
Name: proportion, dtype: object


Flaged Imputated Rows

In [19]:
df['internet_service_imputed'] = df['internet_service'].isna().astype(int)

# (run this BEFORE the fillna cell above, or recompute from original)
# Better pattern — do both in one block:

df2 = pd.read_csv("D:\Data Analysis Projects\CSV files\customer_churn_dataset.csv")   # reload clean copy
df2['internet_service_imputed'] = df2['internet_service'].isna().astype(int)
df2['internet_service'] = df2['internet_service'].fillna(df2['internet_service'].mode()[0])

print(f"Imputed flag — 1s: {df2['internet_service_imputed'].sum()}, 0s: {(df2['internet_service_imputed']==0).sum()}")

Imputed flag — 1s: 2013, 0s: 17987


In [5]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

Duplicate rows: 0


Summary Statistics

In [6]:
print("=== Numeric columns ===")
display(df.describe())

print("\n=== Categorical columns ===")
display(df.describe(include='object'))

=== Numeric columns ===


,customer_id,tenure,monthly_charges,total_charges,support_calls
count,20000.00,20000.00,20000.00,20000.00,20000.00
mean,10000.50,36.47,70.01,2543.98,1.51
std,5773.65,20.77,28.89,1882.95,1.24
min,1.00,1.00,20.00,20.23,0.00
25%,5000.75,18.00,45.21,1045.84,1.00
50%,10000.50,36.00,70.09,2096.49,1.00
75%,15000.25,54.00,95.07,3690.34,2.00
max,20000.00,72.00,120.00,8629.92,8.00



=== Categorical columns ===


,contract,payment_method,internet_service,tech_support,online_security,churn
count,20000,20000,17987,20000,20000,20000
unique,3,4,2,2,2,2
top,Month-to-month,Credit,Fiber,No,No,No
freq,11942,5026,10064,13031,12008,13157


In [12]:
print("=" * 45)
print("DATASET SUMMARY")
print("=" * 45)
print(f"  Rows          : {df.shape[0]}")
print(f"  Columns       : {df.shape[1]}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Duplicates    : {df.duplicated().sum()}")
if 'Yes' in df['churn'].values:
    rate = (df['churn'] == 'Yes').mean() * 100
    print(f"  Churn rate    : {rate:.1f}%")
print("=" * 45)

DATASET SUMMARY
  Rows          : 20000
  Columns       : 11
  Missing values: 2013
  Duplicates    : 0
  Churn rate    : 34.2%


Final Dataset Health Check

In [20]:
print("=" * 45)
print("POST-CLEANING HEALTH CHECK")
print("=" * 45)
print(f"  Shape             : {df2.shape}")
print(f"  Missing values    : {df2.isnull().sum().sum()}")
print(f"  Duplicates        : {df2.duplicated().sum()}")
print(f"  Churn rate        : {(df2['churn']=='Yes').mean()*100:.1f}%")
print()
print("=== Dtypes ===")
print(df2.dtypes)
print()
print("=== Categorical value counts ===")
cat_cols = ['contract', 'payment_method', 'internet_service', 'tech_support', 'online_security', 'churn']
for col in cat_cols:
    print(f"\n{col}:\n{df2[col].value_counts().to_string()}")

POST-CLEANING HEALTH CHECK
  Shape             : (20000, 12)
  Missing values    : 0
  Duplicates        : 0
  Churn rate        : 34.2%

=== Dtypes ===
customer_id                   int64
tenure                        int64
monthly_charges             float64
total_charges               float64
contract                     object
payment_method               object
internet_service             object
tech_support                 object
online_security              object
support_calls                 int64
churn                        object
internet_service_imputed      int64
dtype: object

=== Categorical value counts ===

contract:
contract
Month-to-month    11942
One year           4990
Two year           3068

payment_method:
payment_method
Credit    5026
Debit     5025
Cash      4995
UPI       4954

internet_service:
internet_service
Fiber    12077
DSL       7923

tech_support:
tech_support
No     13031
Yes     6969

online_security:
online_security
No     12008
Yes     7992

ch

In [21]:
df2.to_csv("D:\Data Analysis Projects\CSV files\Cleaned CSV\Telecom_Churn_Clean.csv", index=False)
print("Saved → telecom_churn_clean.csv")

Saved → telecom_churn_clean.csv
